# 1 Once verilerimizi alalim

In [ ]:
import requests
import zipfile
from pathlib import Path

# Once Folder i kuruyoruz
data_path = Path("Benim Datam/")
image_path = data_path / "Duygular"

if image_path.is_dir():
  print("Dosya zaten var.")
else:
  print("Dosya olusturuluyor.")
  image_path.mkdir(parents=True, exist_ok=True)

# Url ile verimizi alalim
url = " https://www.kaggle.com/api/v1/datasets/download/samithsachidanandan/human-face-emotions"

with open(data_path / "Duygular.zip", "wb") as f:
  response = requests.get(url)
  print("Downloading...")
  f.write(response.content)

# Unzipden kurtaralim
with zipfile.ZipFile(data_path / "Duygular.zip", "r") as zip_ref:
  print("Unzipping...")
  zip_ref.extractall(image_path)

In [ ]:
import os
for image_path, dirnames, filenames in os.walk(image_path):
  print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{image_path}'.")

In [ ]:
import os
import shutil
import random

# Ana klasör ve hedef klasör
src_dir = "Benim Datam/Duygular/Data"
dest_dir = "Benim Datam/Duygular/dataset"

# Train/Test oranı
train_ratio = 0.8

# Sınıfları al
classes = [d for d in os.listdir(src_dir) if os.path.isdir(os.path.join(src_dir, d))]

for cls in classes:
    cls_path = os.path.join(src_dir, cls)
    images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    random.shuffle(images)

    split_idx = int(len(images) * train_ratio)
    train_imgs = images[:split_idx]
    test_imgs = images[split_idx:]

    # Train klasörüne kopyala
    train_class_dir = os.path.join(dest_dir, "train", cls)
    os.makedirs(train_class_dir, exist_ok=True)
    for img in train_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(train_class_dir, img))

    # Test klasörüne kopyala
    test_class_dir = os.path.join(dest_dir, "test", cls)
    os.makedirs(test_class_dir, exist_ok=True)
    for img in test_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(test_class_dir, img))

print("Train/Test ayrımı tamamlandı!")

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
image_path = image_path / "dataset"

In [ ]:
train_dir = image_path / "train"
test_dir = image_path / "test"

train_dir, test_dir

In [ ]:
from PIL import Image

# 1. Tum goruntuleri bir list yapalim
image_path_list = list(image_path.glob("*/*/*.jpg"))

# 2. Pick a random image path
random_image_path = random.choice((image_path_list))

# 3. Get image class from path name
image_class = random_image_path.parent.stem

# 4. Open image
img = Image.open(random_image_path)

# 5. Print metadata
print(f"Random image path: {random_image_path}")
print(f"Image class: {image_class}")
print(f"Image height: {img.height}")
print(f"Image width: {img.width}")
img

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

data_trasformer = transforms.Compose([
    transforms.Resize(size=(64,64)),
    transforms.ToTensor(),
    transforms.Grayscale(num_output_channels=1)
])

train_data = datasets.ImageFolder(root=train_dir,
                                  transform=data_trasformer)
test_data = datasets.ImageFolder(root=test_dir,
                                 transform=data_trasformer)
class_names = train_data.classes
class_names

In [ ]:
img, label = train_data[0][0], train_data[0][1]
img

In [ ]:
label

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(batch_size=32,
                              shuffle=True,
                              num_workers=2,
                              dataset=train_data)
test_dataloader = DataLoader(batch_size=32,
                             num_workers=2,
                             shuffle=False,
                             dataset=test_data)
train_dataloader, test_dataloader

In [ ]:
img, label = next(iter(train_dataloader))
print(f"Image shape: {img.shape} -> [batch_size, color_channels, height, width]")
print(f"Label shape: {label.shape}")

In [ ]:
import torch
from torch import nn
class Model_1(nn.Module):
  def __init__(self):
     super().__init__()

     self.blok_1 = nn.Sequential(
        nn.Conv2d(
             in_channels=1,
             out_channels=16,
             kernel_size=3,
             padding=1,
             stride=1
         ),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.Conv2d(
             in_channels=16,
             out_channels=16,
             kernel_size=3,
             padding=1,
             stride=1
         ),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     stride=2)
)
     self.blok_2 = nn.Sequential(
        nn.Conv2d(
             in_channels=16,
             out_channels=16,
             kernel_size=3,
             padding=1,
             stride=1
         ),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.Conv2d(
             in_channels=16,
             out_channels=16,
             kernel_size=3,
             padding=1,
             stride=1
         ),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     stride=2)
)
     self.classifier = nn.Sequential(
          nn.Flatten(),
          nn.Linear(in_features=16*16*16,
                    out_features=16*16*16),
          nn.ReLU(),
          nn.Linear(in_features=16*16*16,
                    out_features=5)
      )
  def forward(self,x):
    x = self.blok_1(x)
    #print(x.shape)
    x = self.blok_2(x)
    #print(x.shape)
    x = self.classifier(x)
    #print(x.shape)

    return x

In [ ]:
model_1 = Model_1()

In [ ]:
model_1

In [ ]:
image_batch, label_batch = next(iter(train_dataloader))
image_batch.shape, label_batch.shape

In [ ]:
model_1(image_batch)

In [ ]:
!pip install torchinfo
import torchinfo
from torchinfo import summary
from timeit import default_timer as timer
from tqdm import tqdm
summary(model_1, input_size=(32,1,64,64))

In [ ]:
def train_step(model,
              dataloader,
              loss_fn,
              optimizer,
              device):

  model.train()

  train_loss, train_acc = 0, 0

  for batch, (X, y) in enumerate(dataloader):
    X = X.to(device)
    y = y.to(device)
    y_pred = model(X)

    loss = loss_fn(y_pred, y)
    train_loss += loss.item()

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
    train_acc += (y_pred_class==y).sum().item()/len(y_pred)

  train_loss = train_loss / len(dataloader)
  train_acc = train_acc / len(dataloader) *100
  return train_loss, train_acc

In [ ]:
def test_step(model,
              dataloader,
              loss_fn,
              device):
  model.eval()

  test_loss, test_acc = 0, 0
  with torch.inference_mode():
    for batch, (X,y) in enumerate(dataloader):
      X = X.to(device)
      y = y.to(device)
      test_pred_logits = model(X)

      loss = loss_fn(test_pred_logits,y)
      test_loss += loss.item()

      test_pred_labels = test_pred_logits.argmax(dim=1)
      test_acc += (test_pred_labels==y).sum().item()/len(test_pred_labels)
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader) *100
    return test_loss, test_acc

In [ ]:
def train(model,
          train_data,
          test_data,
          loss_fn,
          optimizer,
          epochs,
          device):

  results = {"train_loss": [],
           "train_acc": [],
           "test_loss":[],
           "test_acc":[]}
  for epoch in tqdm(range(epochs)):
    train_loss, train_acc = train_step(model=model.to(device),
                                       dataloader=train_dataloader,
                                       loss_fn=loss_fn,
                                       optimizer=optimizer,
                                       device=device)
    test_loss, test_acc = test_step(model=model.to(device),
                                    dataloader=test_dataloader,
                                    loss_fn=loss_fn,
                                    device=device)

    print(f"Epoch: {epoch} | Train loss: {train_loss:.4f} | Train acc: {train_acc:.2f}% | Test loss: {test_loss:.4f} | Test acc: {test_acc:.2f}%")

    results["train_loss"].append(train_loss)
    results["train_acc"].append(train_acc)
    results["test_loss"].append(test_loss)
    results["test_acc"].append(test_acc)

  return results

In [ ]:
# Set random seed
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Set epochs

NUM_EPOCHS = 10

# Recreate an instance of TinyVGG
model_1 = Model_1()

# Setup loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_1.parameters(),
                             lr=0.001)

# Start the timer
start_time = timer()

# Train model_0
model_1_results = train(model=model_1,
                        train_data=train_dataloader,
                        test_data=test_dataloader,
                        optimizer=optimizer,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS,
                        device=device)

# End the timer
end_time = timer()
print(f"Total trainin time: {end_time-start_time:.3f} seconds")

In [ ]:
torch.save(model_1, "full_model_1.pth")

In [ ]:
def plot_loss_curves(results):
  # Get the loss values of the results dictionary
  loss = results["train_loss"]
  test_loss = results["test_loss"]

  # Get the accuracy values of the results dictionary
  accuracy = results["train_acc"]
  test_accuracy = results["test_acc"]

  # Figure out how many epochs there were
  epochs = range(len(results["train_loss"]))

  # Setup a plot
  plt.figure(figsize=(15,7))

  # Plot the loss
  plt.subplot(1,2,1)
  plt.plot(epochs, loss, label="train_loss")
  plt.plot(epochs, test_loss, label="test_loss")
  plt.title("Loss")
  plt.xlabel("Epochs")
  plt.legend();

  # Plot the accuracy
  plt.subplot(1,2,2)
  plt.plot(epochs, accuracy, label="train_accuracy")
  plt.plot(epochs, test_accuracy, label="test_accuracy")
  plt.title("Accuracy")
  plt.xlabel("Epochs")
  plt.legend();

In [ ]:
import matplotlib.pyplot as plt

plot_loss_curves(model_1_results)

In [ ]:
def tahmin(url, transforms, file_name: str, model):
  # Setup custom image path
  custom_image_path = data_path / file_name

  # Download the image if it doesn't already exist
  if not custom_image_path.is_file():
    with open(custom_image_path, "wb") as f:
      # When downloading from GitHub, need to use the "raw" file link
      request = requests.get(url)
      print("Image downloaded")
      f.write(request.content)

  custom_image_unit8 = torchvision.io.read_image(str(custom_image_path)).type(torch.float32) / 255
  print(f"Shape: {custom_image_unit8.shape}")
  print(f"Type: {custom_image_unit8.dtype}")

  custom_image_transformed = transforms(custom_image_unit8)
  print(f"Shape: {custom_image_transformed.shape}")

  model_1.eval()
  with torch.inference_mode():
    custom_pred = model_1(custom_image_transformed.unsqueeze(dim=0).to(device))
  print(f"Shape: {custom_image_transformed.shape}")

  custom_pred_probs = torch.softmax(custom_pred,dim=1)

  the_pred = class_names[custom_pred_probs.argmax(dim=1)]

  return the_pred

In [ ]:
import torchvision
custom_image_transform = transforms.Compose([
    transforms.Resize(size=(64,64)),
    transforms.Grayscale(num_output_channels=1)
])

In [ ]:
a = tahmin(model=model_1, transforms=custom_image_transform, file_name="mucdss.jpg", url="https://www.polonyaakademi.com/wp-content/uploads/2019/06/K%C4%B1zg%C4%B1nl%C4%B1%C4%9F%C4%B1n-%C4%B0lm%C3%AE-Ve-Ameli-%C4%B0l%C3%A2c%C4%B1-1.jpg")
print(f"Tahmin: {a}")

In [ ]:
# Import tqdm.auto
from tqdm.auto import tqdm

# 1. Make predictions with trained model
y_preds = []
model_1.eval()
with torch.inference_mode():
  for x, y in tqdm(test_dataloader, desc="Making predictions ... "):
     x = x.to(device)
     y = y.to(device)
     # Do the forward pass
     y_logit = model_1(x).to(device)
     # Turn predictions from logits -> prediction probabilities -> prediction labels
     y_pred = torch.softmax(y_logit.squeeze(), dim=0).argmax(dim=1)
     # Put prediction on CPU for evaluation
     y_preds.append(y_pred)
  # Concatenate list of predictions into a tensor #
  #print(y_preds)
  y_pred_tensor = torch.cat(y_preds)
  y_pred_tensor

In [ ]:
import torch
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix
import matplotlib.pyplot as plt

# Pred ve target tensor haline getir
y_pred_tensor = torch.tensor(y_pred_tensor) if isinstance(y_pred_tensor, list) else y_pred_tensor
target_tensor = torch.tensor(test_data.targets) if isinstance(test_data.targets, list) else test_data.targets

# GPU’da ise CPU’ya al
y_pred_tensor = y_pred_tensor.cpu()
target_tensor = target_tensor.cpu()

# Eğer y_pred logits ise sınıf indexi al
if y_pred_tensor.dim() > 1 and y_pred_tensor.size(1) > 1:
    y_pred_tensor = torch.argmax(y_pred_tensor, dim=1)

# Confusion matrix hesapla
confmat = ConfusionMatrix(task="multiclass", num_classes=len(class_names))
confmat_tensor = confmat(preds=y_pred_tensor, target=target_tensor)

# Plot
fig, ax = plot_confusion_matrix(
    conf_mat=confmat_tensor.numpy(),
    class_names=class_names,
    figsize=(10,7)
)
plt.show()


In [ ]:
# Pred ve target tensor haline getir
y_pred_tensor = torch.tensor(y_pred_tensor) if isinstance(y_pred_tensor, list) else y_pred_tensor
target_tensor = torch.tensor(test_data.targets) if isinstance(test_data.targets, list) else test_data.targets

# GPU’da ise CPU’ya al
y_pred_tensor = y_pred_tensor.cpu()
target_tensor = target_tensor.cpu()

# Eğer y_pred logits ise sınıf indexi al
if y_pred_tensor.dim() > 1 and y_pred_tensor.size(1) > 1:
    y_pred_tensor = torch.argmax(y_pred_tensor, dim=1)

# Confusion matrix hesapla
confmat = ConfusionMatrix(task="multiclass", num_classes=len(class_names))
confmat_tensor = confmat(preds=y_pred_tensor, target=target_tensor)

# Plot
fig, ax = plot_confusion_matrix(
    conf_mat=confmat_tensor.numpy(),
    class_names=class_names,
    figsize=(10,7)
)
plt.show()